# Pneumonia detection from chest X-rays
I use transfer learning with MobileNetV2.

## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models

## 2. Load the data


In [ ]:
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
data_dir = os.path.join(path, "chest_xray", "chest_xray")

img_size = (224, 224)
batch_size = 32


train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(data_dir, "train"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary",
    validation_split=0.2,
    subset="training",
    seed=123
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(data_dir, "train"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=123
)


test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(data_dir, "test"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds.class_names
print(class_names)  # NORMAL = 0, PNEUMONIA = 1

## 3. Load the pretrained model


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

## 4. Build the model


In [ ]:
model = models.Sequential([
    layers.Input(shape=img_size + (3,)),
    layers.Rescaling(1./127.5, offset=-1),

    base_model,

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")  # 0 = normal, 1 = pneumonia
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### 4.2 Class Weight

In [ ]:
num_normal = len(os.listdir(os.path.join(data_dir, "train", "NORMAL")))
num_pneumonia = len(os.listdir(os.path.join(data_dir, "train", "PNEUMONIA")))
total = num_normal + num_pneumonia

class_weight = {
    0: total / (2 * num_normal),
    1: total / (2 * num_pneumonia)
}
print(num_normal, num_pneumonia)
print(class_weight)

## 5. Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight
)

### 5.2 Treshold or Cutoff

In [ ]:
cutoff = 0.9

weights, bias = model.layers[-1].get_weights()
bias = bias - np.log(cutoff / (1 - cutoff))
model.layers[-1].set_weights([weights, bias])

## 6. Training curves

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label="Train")
plt.plot(epochs, val_acc, label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label="Train")
plt.plot(epochs, val_loss, label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.show()

## 7. Test the model

In [ ]:
cutoff = 0.5   # put the cutoff you picked here

# predictions for the whole test set
y_pred = (model.predict(test_ds) > cutoff).astype(int).flatten()

# real labels
y_true = np.concatenate([labels.numpy() for images, labels in test_ds]).flatten()

# accuracy with my cutoff
test_acc = np.mean(y_pred == y_true)
print("Test accuracy:", round(test_acc, 4))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

## 8. Try one image

In [ ]:
folder = os.path.join(data_dir, "test", "PNEUMONIA")
sample_path = os.path.join(folder, sorted(os.listdir(folder))[4])

img = tf.keras.utils.load_img(sample_path, target_size=img_size)
plt.imshow(img)
plt.axis("off")
plt.show()

x = np.expand_dims(tf.keras.utils.img_to_array(img), axis=0)
score = model.predict(x)[0][0]

print("Score:", round(float(score), 4))
if score > 0.5:
    print("Result: Pneumonia")
else:
    print("Result: Normal")

## 9. Export the model for the Streamlit app

In [ ]:
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

model_path = project_dir / "models" / "chestscan_tl.keras"
model_path.parent.mkdir(exist_ok=True)
model.save(model_path)
print(f"Saved model to {model_path}")